# 🔭 VisionaryX — Your Free AI Agent (No API Key Needed)

VisionaryX runs **100% locally**. No OpenAI / Gemini / Anthropic key required.

**What it can do**
- 🧮 Safe calculator
- 🕒 Date & time
- 🌐 Web search (DuckDuckGo, no key)
- 🗒️ Short-term notes (`remember ...` / `recall`)
- 💬 General chat using a small open-source model (Qwen) from Hugging Face

**How to run:** `Run All` ▶️ then chat in the last cell.
If the model can't be downloaded (low RAM / offline), the agent still works in **tools-only mode**.

## 1️⃣ Install dependencies

In [ ]:
# Run once. (Safe to re-run.)
%pip install -q -r requirements.txt

## 2️⃣ Config

In [ ]:
import ast, operator, datetime, re, warnings
warnings.filterwarnings("ignore")

# Small open model, downloads once from Hugging Face (no key needed).
# Low RAM?  use "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
USE_LLM = True          # set False for tools-only mode (instant, no download)
MAX_NEW_TOKENS = 300

## 3️⃣ Tools

In [ ]:
# ---------- Calculator (safe, no eval) ----------
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Mod: operator.mod, ast.FloorDiv: operator.floordiv,
    ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos,
}

def _eval(node):
    if isinstance(node, ast.Expression):
        return _eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        left, right = _eval(node.left), _eval(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 1000:
            raise ValueError("Exponent too large")
        return _OPS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("Unsupported expression")

def calculator(expr: str) -> str:
    try:
        result = _eval(ast.parse(expr.replace("^", "**").strip(), mode="eval"))
        return f"{expr.strip()} = {round(result, 10) if isinstance(result, float) else result}"
    except ZeroDivisionError:
        return "Cannot divide by zero."
    except Exception as e:
        return f"Could not calculate that ({e})."

# ---------- Date & time ----------
def get_time(_: str = "") -> str:
    return datetime.datetime.now().strftime("It's %A, %d %B %Y, %I:%M %p.")

# ---------- Web search (DuckDuckGo, no key) ----------
def web_search(query: str, n: int = 3) -> str:
    try:
        try:
            from ddgs import DDGS
        except ImportError:
            from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            hits = list(ddgs.text(query, max_results=n))
        if not hits:
            return "No results found."
        return "\n".join(f"- {h['title']}: {h['body']} ({h['href']})" for h in hits)
    except Exception as e:
        return f"Web search unavailable right now ({type(e).__name__}). Check your internet connection."

# ---------- Notes (in-memory) ----------
NOTES = []

def remember(text: str) -> str:
    NOTES.append(text.strip())
    return f"Got it. I'll remember: \"{text.strip()}\""

def recall(_: str = "") -> str:
    if not NOTES:
        return "I don't have any notes yet. Say: remember <something>"
    return "Here's what I have:\n" + "\n".join(f"{i}. {n}" for i, n in enumerate(NOTES, 1))

TOOLS = {
    "calculator": calculator, "get_time": get_time, "web_search": web_search,
    "remember": remember, "recall": recall,
}

## 4️⃣ Router — decides which tool to use

In [ ]:
def route(msg: str):
    """Returns (tool_name, argument) or (None, None) for normal chat."""
    m = msg.strip()
    low = m.lower()

    # math like "12*(3+4)" or "calculate 2^10"
    mm = re.fullmatch(r"(?:calc(?:ulate)?|what is|what's|solve)?\s*([\d\s\.\+\-\*/\^\(\)%]+)\??", low)
    if mm and re.search(r"\d", mm.group(1)) and re.search(r"[\+\-\*/\^%]", mm.group(1)):
        return "calculator", mm.group(1)

    if re.search(r"what('s| is)? the (current )?(time|date)\b|\bcurrent (time|date)\b|\btime now\b|\btoday'?s date\b|\bwhat day is it\b", low):
        return "get_time", ""

    if low.startswith("remember "):
        return "remember", m[len("remember "):]
    if low in {"recall", "what do you remember", "show notes", "notes"}:
        return "recall", ""

    s = re.match(r"(?:search(?: for)?|google|look ?up|latest news about|news about|find)\s+(.+)", low)
    if s:
        return "web_search", s.group(1)

    return None, None

## 5️⃣ Brain — local open-source LLM (no key)

In [ ]:
class Brain:
    def __init__(self):
        self.ready = False
        if not USE_LLM:
            print("ℹ️  Tools-only mode (USE_LLM=False).")
            return
        try:
            import torch
            from transformers import AutoTokenizer, AutoModelForCausalLM
            print(f"⏳ Loading {MODEL_NAME} (first run downloads it) ...")
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
            self.tok = AutoTokenizer.from_pretrained(MODEL_NAME)
            self.model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto").to(self.device)
            self.ready = True
            print(f"✅ Brain ready on {self.device}.")
        except Exception as e:
            print(f"⚠️  Could not load the model ({type(e).__name__}: {e}). Running in tools-only mode.")

    def generate(self, messages):
        text = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tok([text], return_tensors="pt").to(self.device)
        out = self.model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True, temperature=0.7, top_p=0.9,
        )
        return self.tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

## 6️⃣ The Agent

In [ ]:
SYSTEM_PROMPT = (
    "You are VisionaryX, a friendly and concise AI assistant. "
    "Answer clearly in a few sentences. If the user writes in Tamil (Tanglish) or another language, reply in that style. "
    "If you are unsure, say so instead of guessing."
)

class VisionaryX:
    def __init__(self):
        self.brain = Brain()
        self.history = []   # last few chat turns

    def run(self, user_msg: str) -> str:
        tool, arg = route(user_msg)

        # 1) Tool path
        if tool:
            result = TOOLS[tool](arg)
            if tool == "web_search" and self.brain.ready:
                msgs = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"Question: {user_msg}\n\nSearch results:\n{result}\n\n"
                                                 "Summarize the answer briefly using only these results."},
                ]
                return self.brain.generate(msgs) + "\n\n🔗 Sources:\n" + result
            return result

        # 2) Chat path
        if not self.brain.ready:
            return ("I'm in tools-only mode right now. Try: 'calculate 12*8', 'what is the time', "
                    "'search python 3.13', 'remember buy milk', or set USE_LLM=True for full chat.")

        msgs = [{"role": "system", "content": SYSTEM_PROMPT}] + self.history[-6:] + [
            {"role": "user", "content": user_msg}
        ]
        reply = self.brain.generate(msgs)
        self.history += [{"role": "user", "content": user_msg}, {"role": "assistant", "content": reply}]
        return reply

agent = VisionaryX()

## 7️⃣ Quick demo

In [ ]:
for q in ["calculate 12*(3+4)", "what is the time", "remember my exam is on Friday", "recall"]:
    print("You:", q)
    print("VisionaryX:", agent.run(q), "\n")

## 8️⃣ Chat with VisionaryX 💬
Type `exit` to stop.

In [ ]:
def chat():
    print("VisionaryX is ready! Type 'exit' to quit.\n")
    while True:
        user = input("You: ").strip()
        if user.lower() in {"exit", "quit", "bye"}:
            print("VisionaryX: Bye! 👋")
            break
        if user:
            print("VisionaryX:", agent.run(user), "\n")

chat()